# 🔍 Long Châu Scraper — Bước 1: Khám Phá URL (URL Discovery qua A-Z)

Notebook này thực hiện quét toàn bộ danh mục thuốc tra cứu theo chữ cái từ A đến Z của Nhà Thuốc Long Châu (`nhathuoclongchau.com.vn`), tự động tính số trang và lưu danh sách URL vào file hàng đợi `url_queue_longchau.json` (**Resume-Safe**).

In [ ]:
import sys
import time
import math
import json
import re
import random
from pathlib import Path
from datetime import datetime, timezone
import requests
from tqdm.notebook import tqdm

from config import (
    AZ_INDEX_URL,
    BASE_URL,
    MAX_RETRIES,
    QUEUE_FILE,
    REQUEST_DELAY_MAX,
    REQUEST_DELAY_MIN,
    TIMEOUT_SECONDS,
    USER_AGENTS,
)

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

In [ ]:
def load_queue():
    if QUEUE_FILE.exists():
        try:
            with open(QUEUE_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f'⚠️ Warning: Lỗi đọc file queue ({e}), khởi tạo mới.')
    return {}

def save_queue(queue):
    with open(QUEUE_FILE, 'w', encoding='utf-8') as f:
        json.dump(queue, f, ensure_ascii=False, indent=2)

def get_headers():
    return {
        'User-Agent': random.choice(USER_AGENTS),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
    }

def fetch_az_page(letter: str, page: int = 1):
    url = f'{AZ_INDEX_URL}?alphabet={letter}&page={page}'
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=get_headers(), timeout=TIMEOUT_SECONDS)
            if resp.status_code == 200:
                match = re.search(r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>', resp.text, re.DOTALL)
                if match:
                    data = json.loads(match.group(1))
                    page_props = data.get('props', {}).get('pageProps', {})
                    return page_props.get('listDrugs', {})
        except Exception as e:
            pass
        time.sleep(attempt * 1.2)
    return None

def discover_urls(letters=None):
    if not letters:
        letters = [chr(i) for i in range(ord('A'), ord('Z') + 1)]
    
    queue = load_queue()
    initial_count = len(queue)
    new_urls_count = 0

    print(f'🔍 Bắt đầu khám phá URL cho danh sách chữ cái: {", ".join(letters)}')
    print(f'📋 Kích thước Queue hiện tại: {initial_count} items')

    for letter in letters:
        print(f'\n---> Quét chữ cái: "{letter}"')
        first_page_data = fetch_az_page(letter, page=1)
        if not first_page_data:
            print(f'❌ Không thể lấy trang 1 cho chữ cái "{letter}". Bỏ qua.')
            continue
        
        total_count = first_page_data.get('totalCount', 0)
        items = first_page_data.get('items', [])
        total_pages = math.ceil(total_count / 20) if total_count > 0 else (1 if items else 0)
        print(f'  📊 Tổng số thuốc tìm thấy: {total_count} ({total_pages} trang)')

        for item in items:
            slug = item.get('slug')
            if slug:
                full_url = f'{BASE_URL}/{slug.lstrip("/")}' if not slug.startswith('http') else slug
                if full_url not in queue:
                    queue[full_url] = {
                        'status': 'pending',
                        'discovered_at': datetime.now(timezone.utc).isoformat(),
                        'name': item.get('webName') or item.get('name') or '',
                        'attempts': 0,
                        'error': None
                    }
                    new_urls_count += 1

        if total_pages > 1:
            pbar = tqdm(range(2, total_pages + 1), desc=f'  Trang cho "{letter}"', unit='page')
            for page in pbar:
                time.sleep(random.uniform(REQUEST_DELAY_MIN, REQUEST_DELAY_MAX))
                page_data = fetch_az_page(letter, page=page)
                if page_data:
                    for item in page_data.get('items', []):
                        slug = item.get('slug')
                        if slug:
                            full_url = f'{BASE_URL}/{slug.lstrip("/")}' if not slug.startswith('http') else slug
                            if full_url not in queue:
                                queue[full_url] = {
                                    'status': 'pending',
                                    'discovered_at': datetime.now(timezone.utc).isoformat(),
                                    'name': item.get('webName') or item.get('name') or '',
                                    'attempts': 0,
                                    'error': None
                                }
                                new_urls_count += 1
                if page % 5 == 0:
                    save_queue(queue)
        save_queue(queue)

    print('\n==========================================')
    print('✅ Hoàn thành khám phá URL!')
    print(f'➕ URL mới thêm vào: {new_urls_count}')
    print(f'📋 Tổng số URL trong Queue: {len(queue)}')
    print(f'💾 Queue lưu tại: {QUEUE_FILE}')

In [ ]:
# Chạy thử nghiệm quét cho chữ cái 'A' (truyền None để quét tất cả A-Z)
discover_urls(letters=['A'])